In [2]:
import sys
sys.path.append('../../')
from tqdm import tqdm
import os
import torch
import pandas as pd
import torch.nn.functional as F
import numpy as np
from utilities import load_embedding, EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from datetime import datetime
import pickle
from utilities import print_exams

class fluProfiler_Dataset(Dataset):
    def __init__(self, DataFrame):
        self.emb_file_name_a = ('matrix_' + DataFrame['seq_id_a']).tolist()
        self.emb_file_name_b = ('matrix_' + DataFrame['seq_id_b']).tolist()
        self.emb_file_name_c = ('matrix_' + DataFrame['seq_id_c']).tolist()
        self.emb_file_name_d = ('matrix_' + DataFrame['seq_id_d']).tolist()

        self.strainPassCats = convert_Pass2tensor(('<cls>' + DataFrame['serumPassCat'] + '<eos>' + DataFrame['virusPassCat'] + '<eos>').tolist())

        self.labels = torch.tensor(DataFrame['label'].tolist())
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.emb_file_name_a[idx], self.emb_file_name_b[idx], self.emb_file_name_c[idx], self.emb_file_name_d[idx], \
               self.strainPassCats[idx], self.labels[idx]

def convert_Pass2tensor(pass_cats):
    result = [
        item.replace('<cls>', '0').replace('<eos>', '1').replace('<EGG>', '2').replace('<CELL>', '3').replace('<BOTH>', '4')
        for item in pass_cats
    ]
    result = torch.tensor([[int(number) for number in [char for char in item]] for item in result])
    return result

def list2df(mylist, period):
    merged_rows = []
    for i in range(0, len(mylist), period):
        merged_row = []
        for j in range(period):
            merged_row += mylist[i + j]
        merged_rows.append(merged_row)

    return pd.DataFrame(merged_rows)

def generate_matrix(matrix_list):
    seq_len = [mat.shape[0] for mat in matrix_list]
    max_len = max(seq_len)
    mask_list = []
    for i in range(len(matrix_list)): 
        matrix_list[i] = F.pad(matrix_list[i], (0, 0, 0, max_len - seq_len[i]))
        mask = torch.concat((torch.ones(1,seq_len[i]),torch.zeros(1,max_len-seq_len[i])),axis=1)
        mask_list.append(mask)
    matrix = torch.stack(matrix_list)
    mask = torch.stack(mask_list).view(len(matrix_list),max_len)
    return matrix, mask

device = torch.device('cuda:2')

In [4]:
test_data = pd.read_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/data/data_40/serum/test.csv')
test_dataset = fluProfiler_Dataset(test_data)
test_dataloader = DataLoader(test_dataset, batch_size=100, shuffle=False)

In [8]:
import os
from tqdm import tqdm

embedding_df = test_data
# load embedding
sequence_names = pd.concat([embedding_df['seq_id_a'], embedding_df['seq_id_b'], 
                            embedding_df['seq_id_c'], embedding_df['seq_id_d']]).unique().tolist()
sequence_names = ['matrix_' + item + '.pt' for item in sequence_names]
emb_dict = load_embedding("/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/data/data_40/embedding_Crick", files=sequence_names)

Loading tensor: 100%|██████████| 7035/7035 [18:39<00:00,  6.29file/s]


In [10]:
emb_dict = {key: value.cpu() for key, value in emb_dict.items()}

In [ ]:
model = torch.load(f='/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/trained_model/1.9_cp/2025-08-05_14-49-58.pth', weights_only=False, map_location=device)

prediction_ls_test = []
reference_ls_test = []
logits_ls = []
loss_ls_test = []
model.eval()
for batch in test_dataloader:
    emb_file_name_a, emb_file_name_b, emb_file_name_c, emb_file_name_d, strainPassCats, labels = batch

    matrixs_a, masks_a = generate_matrix([emb_dict[key] for key in emb_file_name_a])
    matrixs_b, masks_b = generate_matrix([emb_dict[key] for key in emb_file_name_b])
    matrixs_c, masks_c = generate_matrix([emb_dict[key] for key in emb_file_name_c])
    matrixs_d, masks_d = generate_matrix([emb_dict[key] for key in emb_file_name_d])

    matrixs_a, matrixs_b, matrixs_c, matrixs_d = matrixs_a.to(device), matrixs_b.to(device), matrixs_c.to(device), matrixs_d.to(device)
    masks_a = masks_a.to(device)
    masks_b = masks_b.to(device)
    masks_c = masks_c.to(device)
    masks_d = masks_d.to(device)

    strainPassCats = strainPassCats.to(device)

    labels = labels.to(device)
    with torch.no_grad():
        loss, logits, output = model(matrices_a=matrixs_a, matrices_b=matrixs_b, matrices_c=matrixs_c,
                                        matrices_d=matrixs_d, matrix_attention_masks_a=masks_a, matrix_attention_masks_b=masks_b,
                                        matrix_attention_masks_c=masks_c, matrix_attention_masks_d=masks_d, strainPassCats=strainPassCats,
                                        labels=labels)

    loss_ls_test.append(loss.item())
    logits_ls.append(logits.tolist())
    prediction_ls_test.extend(output.view(-1).tolist())
    reference_ls_test.extend(labels.tolist())

test_mae, test_mse, test_pearson, test_spearman, test_R2 = print_exams(reference_ls_test, prediction_ls_test)

MAE: 0.61135
MSE: 0.59326
pearson correlation: 0.90443
spearman correlation: 0.82248
R2_score: 0.76888


: 

In [6]:
from model_001 import LucaQuadruple_final_dropout

In [4]:
model = torch.load(f='/data/chenyihao/fluProfiler_source/trained_model/1.9/2025-07-30_13-11-53.pth', weights_only=False, map_location=device)

prediction_ls = []
reference_ls = []
logits_ls = []
loss_ls_valid = []
model.eval()
for batch in test_dataloader:
    emb_file_name_a, emb_file_name_b, emb_file_name_c, emb_file_name_d, strainPassCats, labels = batch
    
    matrixs_a, masks_a = generate_matrix([emb_dict[key] for key in emb_file_name_a])
    matrixs_b, masks_b = generate_matrix([emb_dict[key] for key in emb_file_name_b])
    matrixs_c, masks_c = generate_matrix([emb_dict[key] for key in emb_file_name_c])
    matrixs_d, masks_d = generate_matrix([emb_dict[key] for key in emb_file_name_d])

    matrixs_a, matrixs_b, matrixs_c, matrixs_d = matrixs_a.to(device), matrixs_b.to(device), matrixs_c.to(device), matrixs_d.to(device)
    masks_a = masks_a.to(device)
    masks_b = masks_b.to(device)
    masks_c = masks_c.to(device)
    masks_d = masks_d.to(device)

    strainPassCats = strainPassCats.to(device)

    labels = labels.to(device)
    with torch.no_grad():
        loss, logits, output = model(matrices_a=matrixs_a, matrices_b=matrixs_b, matrices_c=matrixs_c, 
                                        matrices_d=matrixs_d, matrix_attention_masks_a=masks_a, matrix_attention_masks_b=masks_b, 
                                        matrix_attention_masks_c=masks_c, matrix_attention_masks_d=masks_d, strainPassCats=strainPassCats, 
                                        labels=labels)

    loss_ls_valid.append(loss.item())
    logits_ls.append(logits.tolist())
    prediction_ls = prediction_ls + output.view(-1).tolist()
    reference_ls = reference_ls + labels.tolist()

print_exams(reference_ls, prediction_ls)

MAE:  0.6966944270764907
MSE:  0.8363919836434318
pearson correlation:  PearsonRResult(statistic=0.8866071323885858, pvalue=0.0)
spearman correlation:  SignificanceResult(statistic=0.8506915160119894, pvalue=0.0)
R2_score:  0.7337786649710172


In [6]:
model = torch.load(f='/data/chenyihao/fluProfiler_source/trained_model/1.9/2025-07-30_15-31-14.pth', weights_only=False, map_location=device)

prediction_ls = []
reference_ls = []
logits_ls = []
loss_ls_valid = []
model.eval()
for batch in test_dataloader:
    emb_file_name_a, emb_file_name_b, emb_file_name_c, emb_file_name_d, strainPassCats, labels = batch
    
    matrixs_a, masks_a = generate_matrix([emb_dict[key] for key in emb_file_name_a])
    matrixs_b, masks_b = generate_matrix([emb_dict[key] for key in emb_file_name_b])
    matrixs_c, masks_c = generate_matrix([emb_dict[key] for key in emb_file_name_c])
    matrixs_d, masks_d = generate_matrix([emb_dict[key] for key in emb_file_name_d])

    matrixs_a, matrixs_b, matrixs_c, matrixs_d = matrixs_a.to(device), matrixs_b.to(device), matrixs_c.to(device), matrixs_d.to(device)
    masks_a = masks_a.to(device)
    masks_b = masks_b.to(device)
    masks_c = masks_c.to(device)
    masks_d = masks_d.to(device)

    strainPassCats = strainPassCats.to(device)

    labels = labels.to(device)
    with torch.no_grad():
        loss, logits, output = model(matrices_a=matrixs_a, matrices_b=matrixs_b, matrices_c=matrixs_c, 
                                        matrices_d=matrixs_d, matrix_attention_masks_a=masks_a, matrix_attention_masks_b=masks_b, 
                                        matrix_attention_masks_c=masks_c, matrix_attention_masks_d=masks_d, strainPassCats=strainPassCats, 
                                        labels=labels)

    loss_ls_valid.append(loss.item())
    logits_ls.append(logits.tolist())
    prediction_ls = prediction_ls + output.view(-1).tolist()
    reference_ls = reference_ls + labels.tolist()

print_exams(reference_ls, prediction_ls)

MAE:  0.7041785097571913
MSE:  0.8519543604965095
pearson correlation:  PearsonRResult(statistic=0.8840244411542455, pvalue=0.0)
spearman correlation:  SignificanceResult(statistic=0.8453142504198075, pvalue=0.0)
R2_score:  0.7168753166971513


In [12]:
pd.read_csv('../../../data/processed/H3N2_42.csv', index_col=False).columns

Index(['seq_id_a', 'seq_id_b', 'seq_id_c', 'seq_id_d', 'seq_a', 'seq_b',
       'seq_c', 'seq_d', 'serumPassCat', 'virusPassCat', 'serumName',
       'virusName', 'virusDate', 'virusIslID', 'serumType', 'HI_Dist'],
      dtype='object')

In [13]:
group_columns = ['seq_a', 'seq_b', 'serumPassCat', 'virusPassCat']
# new_columns = ['seq_id_a', 'seq_id_c','seq_a', 'seq_b', 'serumPassCat', 'virusPassCat', 'serumName', 'virusName', 'label']

Crick_42 = pd.read_csv('../../../data/processed/H3N2_42.csv', index_col=False)[['seq_id_a', 'seq_id_c', 'seq_a','seq_c', 'serumPassCat', 'virusPassCat', 
                                                                                'serumName', 'virusName', 'virusDate', 'virusIslID', 'serumType', 'HI_Dist']]
Crick_42.columns = ['seq_id_a', 'seq_id_b', 'seq_a','seq_b', 'serumPassCat', 'virusPassCat', 'serumName', 'virusName', 'virusDate', 'virusIslID', 'serumType', 'label']
Crick_42_filt1 = Crick_42.groupby(group_columns).agg({'seq_id_a': 'first', 'seq_id_b': 'first', 'serumName': 'first', 'virusName': 'first', 'label': 'mean'}).reset_index()
pass_dict = {'EGG': '<EGG>', 'CELL': '<CELL>'}
Crick_42_final = Crick_42_filt1.copy()
Crick_42_final['serumPassCat'] = Crick_42_final['serumPassCat'].map(pass_dict)
Crick_42_final['virusPassCat'] = Crick_42_final['virusPassCat'].map(pass_dict)

In [14]:
Crick_42_dataset = fluProfiler_Dataset(Crick_42_final)
Crick_42_dataloader = DataLoader(Crick_42_dataset)

In [16]:
import os
from tqdm import tqdm

embedding_df = Crick_42_final
# load embedding
sequence_names = pd.concat([embedding_df['seq_id_a'],embedding_df['seq_id_b']]).unique().tolist()
sequence_names = ['matrix_' + item + '.pt' for item in sequence_names]
IDs, embeddings = load_embedding("/data/chenyihao/embedding_42", files=sequence_names)
future_embeddings = [emb.to(device) for emb in embeddings]
future_emb_dict = dict(zip(IDs, embeddings))

Loading tensor: 100%|██████████| 226/226 [00:16<00:00, 13.54file/s]


In [1]:
model = torch.load(f='/data/chenyihao/fluProfiler_source/trained_model/1.9/2025-07-30_15-04-08.pth', weights_only=False)
model.to(device)

prediction_ls = []
reference_ls = []
logits_ls = []
loss_ls_valid = []
model.eval()
for batch in valid_dataloader:
    emb_file_name_a, emb_file_name_b, emb_file_name_c, emb_file_name_d, strainPassCats, labels = batch
    
    matrixs_a, masks_a = generate_matrix([emb_dict[key] for key in emb_file_name_a])
    matrixs_b, masks_b = generate_matrix([emb_dict[key] for key in emb_file_name_b])
    matrixs_c, masks_c = generate_matrix([emb_dict[key] for key in emb_file_name_c])
    matrixs_d, masks_d = generate_matrix([emb_dict[key] for key in emb_file_name_d])

    matrixs_a, matrixs_b, matrixs_c, matrixs_d = matrixs_a.to(device), matrixs_b.to(device), matrixs_c.to(device), matrixs_d.to(device)
    masks_a = masks_a.to(device)
    masks_b = masks_b.to(device)
    masks_c = masks_c.to(device)
    masks_d = masks_d.to(device)

    strainPassCats = strainPassCats.to(device)

    labels = labels.to(device)
    with torch.no_grad():
        loss, logits, output = model(matrices_a=matrixs_a, matrices_b=matrixs_b, matrices_c=matrixs_c, 
                                        matrices_d=matrixs_d, matrix_attention_masks_a=masks_a, matrix_attention_masks_b=masks_b, 
                                        matrix_attention_masks_c=masks_c, matrix_attention_masks_d=masks_d, strainPassCats=strainPassCats, 
                                        labels=labels)

    loss_ls_valid.append(loss.item())
    logits_ls.append(logits.tolist())
    prediction_ls = prediction_ls + output.view(-1).tolist()
    reference_ls = reference_ls + labels.tolist()

print_exams(reference_ls, prediction_ls)

NameError: name 'torch' is not defined